# YOLO Pollinator Detector

Trains YOLO26n to detect and classify pollinators in full Arctic field images.

**Classes:** `bumblebee` · `fly` · `butterfly` · `other`

CVAT export: **YOLO 1.1** format.

## Small object strategy
Original images are 3008×1692. The smallest insects are ~30×71 px.
Resizing to 640 would shrink them to ~13px — too small for standard YOLO detection.

**Training:** `IMG_SIZE=640` (standard, fast, fits in GPU memory)

**Inference:** SAHI (Slicing Aided Hyper Inference) — slices the full image into
overlapping 640×640 tiles, runs YOLO on each tile, then merges results.
A 30×71px insect appears at ~30px in the full image but at natural size inside its tile.

## Strategy for imbalanced data
fly has more bbox, others have much fewer. We use:
- `copy_paste` augmentation — copies rare-class objects into other images
- `mixup` augmentation — blends images to expose model to rare classes more
- Two-stage training: freeze backbone first, then unfreeze all layers
- External data (optional): mix in Roboflow bumblebee/butterfly images


In [ ]:
# Install ultralytics if needed
# !pip install ultralytics --quiet

import shutil
import json
import random
from pathlib import Path

import numpy as np
import yaml
from ultralytics import YOLO

print('Ultralytics version:', __import__('ultralytics').__version__)


## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATASET_ROOT  = Path('dataset')
EXTRA_DATA_ROOT = None   # e.g. Path('roboflow_bees')
MODEL_OUT_DIR = Path('runs/pollinator')

# ── Classes ────────────────────────────────────────────────────────────────
# Must match the class order used in CVAT annotation
CLASSES = ['bumblebee', 'fly', 'butterfly', 'other']

# ── Model ──────────────────────────────────────────────────────────────────
# yolo26n = best choice for limited data + small objects
# yolo26s = upgrade here once every class has 500+ bbox
MODEL_SIZE = 'yolo26n.pt'

# ── Training settings ──────────────────────────────────────────────────────
# Train at 640 — fast, fits GPU memory.
# Small insects are handled at inference time via SAHI tiling, not by large imgsz.
IMG_SIZE   = 640
BATCH      = 16    # RTX 5060 16GB handles 640 at batch=16 fine
SEED       = 42
VAL_FRAC   = 0.2

# Stage 1: frozen backbone — protects pretrained features when data is limited
EPOCHS_STAGE1 = 30
LR_STAGE1     = 1e-3
FREEZE_LAYERS = 10

# Stage 2: full fine-tune
EPOCHS_STAGE2 = 70
LR_STAGE2     = 1e-4

# ── Augmentation ───────────────────────────────────────────────────────────
# copy_paste copies rare-class bbox into other images — critical for minority classes
COPY_PASTE = 0.3
MIXUP      = 0.1

# ── SAHI inference settings ────────────────────────────────────────────────
# Slice the 3008x1692 image into overlapping tiles for small object detection
SAHI_SLICE_SIZE    = 640    # tile size (matches training imgsz)
SAHI_OVERLAP       = 0.2    # 20% overlap between tiles
SAHI_CONF          = 0.25   # confidence threshold
SAHI_IOU           = 0.5    # NMS IoU threshold for merging tile results

# ── Verify data ────────────────────────────────────────────────────────────
def count_bboxes(labels_dir):
    counts = {i: 0 for i in range(len(CLASSES))}
    for txt in Path(labels_dir).glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                cls = int(parts[0])
                if cls < len(CLASSES):
                    counts[cls] += 1
    return counts

for split in ('train', 'val'):
    ldir = DATASET_ROOT / 'labels' / split
    idir = DATASET_ROOT / 'images' / split
    if not ldir.exists():
        print(f'WARNING: {ldir} not found')
        continue
    n_imgs = len(list(idir.glob('*.jpg'))) + len(list(idir.glob('*.png')))
    counts = count_bboxes(ldir)
    print(f'{split}: {n_imgs} images')
    for i, cls in enumerate(CLASSES):
        flag = '  ← low, consider Roboflow data' if counts[i] < 200 else ''
        print(f'  {cls:15}: {counts[i]:>5} bbox{flag}')


## Prepare Dataset from CVAT Export

Reads the CVAT YOLO 1.1 export structure:
```
project_6_..._yolo 1.1/
├── obj_train_data/   ← all images + .txt labels
├── obj.names         ← class names
├── obj.data
└── train.txt
```
Splits into train/val, removes `unsure` bbox, and generates `data.yaml`.


In [ ]:
# ── Point this at CVAT export folder ─────────────────────────────────
CVAT_EXPORT_DIR = Path('project_6_..._yolo 1.1')  # rename to match YOLO export folder from CVAT
PREPARED_DIR    = Path('dataset')                  # output — notebook reads from here


def prepare_from_cvat(cvat_dir, out_dir, classes, val_frac, seed):
    cvat_dir = Path(cvat_dir)
    out_dir  = Path(out_dir)

    obj_dir    = cvat_dir / 'obj_train_data'
    names_file = cvat_dir / 'obj.names'

    if not obj_dir.exists():
        raise FileNotFoundError(f'obj_train_data not found in {cvat_dir}')

    # ── Read and verify class names ───────────────────────────────────────
    if names_file.exists():
        cvat_names = [l.strip() for l in names_file.read_text().splitlines() if l.strip()]
        print(f'CVAT classes: {cvat_names}')
        print(f'Expected:     {classes}')
        # Build remap: cvat class index → our class index
        # unsure gets mapped to None (will be dropped)
        remap = {}
        for i, name in enumerate(cvat_names):
            if name in classes:
                remap[i] = classes.index(name)
            elif name == 'unsure':
                remap[i] = None  # drop
            else:
                print(f'WARNING: unknown class "{name}" in obj.names — will be dropped')
                remap[i] = None
        print(f'Class remap: {remap}')
    else:
        print('WARNING: obj.names not found — assuming class order matches CLASSES')
        remap = {i: i for i in range(len(classes))}

    # ── Collect all images ────────────────────────────────────────────────
    all_imgs = sorted(list(obj_dir.glob('*.jpg')) +
                      list(obj_dir.glob('*.JPG')) +
                      list(obj_dir.glob('*.png')))
    print(f'\nFound {len(all_imgs)} images in obj_train_data')

    # ── Train/val split ───────────────────────────────────────────────────
    import random
    random.seed(seed)
    shuffled = all_imgs.copy()
    random.shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * val_frac))
    splits = {'val': shuffled[:n_val], 'train': shuffled[n_val:]}
    print(f'Split: train={len(splits["train"])}  val={len(splits["val"])}')

    # ── Copy files and remap labels ───────────────────────────────────────
    dropped_unsure = 0
    for split, imgs in splits.items():
        img_out = out_dir / 'images' / split
        lbl_out = out_dir / 'labels' / split
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in imgs:
            shutil.copy(img_path, img_out / img_path.name)
            lbl_src = obj_dir / (img_path.stem + '.txt')
            lbl_dst = lbl_out / (img_path.stem + '.txt')
            if lbl_src.exists():
                lines = lbl_src.read_text().strip().splitlines()
                kept = []
                for line in lines:
                    parts = line.strip().split()
                    if not parts: continue
                    orig_cls = int(parts[0])
                    new_cls  = remap.get(orig_cls)
                    if new_cls is None:
                        dropped_unsure += 1
                        continue
                    kept.append(f'{new_cls} {" ".join(parts[1:])}')
                lbl_dst.write_text('\n'.join(kept))
            else:
                # No label = background image, write empty txt
                lbl_dst.write_text('')

    print(f'Dropped {dropped_unsure} unsure bbox')

    # ── Write data.yaml ───────────────────────────────────────────────────
    yaml_path = out_dir / 'data.yaml'
    cfg = {
        'path':  str(out_dir.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'nc':    len(classes),
        'names': classes,
    }
    yaml_path.write_text(yaml.dump(cfg, default_flow_style=False))
    print(f'data.yaml written to {yaml_path}')

    # ── Final bbox count ──────────────────────────────────────────────────
    print('\nFinal bbox count per class (train+val):')
    counts = {i: 0 for i in range(len(classes))}
    for split in ('train', 'val'):
        for txt in (out_dir / 'labels' / split).glob('*.txt'):
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    for i, cls in enumerate(classes):
        flag = '  ← low' if counts[i] < 200 else ''
        print(f'  {cls:15}: {counts[i]:>5} bbox{flag}')

    return yaml_path


YAML_PATH = prepare_from_cvat(CVAT_EXPORT_DIR, PREPARED_DIR, CLASSES, VAL_FRAC, SEED)


## Stage 1 — Frozen Backbone Training

Train only the detection head. Backbone stays frozen to preserve pretrained features.
Good when data is limited — prevents backbone from forgetting general visual features.

In [ ]:
model = YOLO(MODEL_SIZE)
print(f'Model: {MODEL_SIZE}')

results_s1 = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE1,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE1,
    freeze=FREEZE_LAYERS,
    patience=15,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage1_frozen',
    exist_ok=True,
    verbose=True,
)

stage1_best = MODEL_OUT_DIR / 'stage1_frozen' / 'weights' / 'best.pt'
print(f'\nStage 1 done. Best weights: {stage1_best}')
print(f'Stage 1 mAP50: {results_s1.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Stage 2 — Full Fine-Tune

Unfreeze all layers and continue training with a small learning rate.
Adapts the backbone to Arctic field image characteristics.

In [ ]:
# Load best Stage 1 checkpoint
model_s2 = YOLO(str(stage1_best))

results_s2 = model_s2.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE2,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE2,
    freeze=0,           # unfreeze all layers
    patience=20,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage2_finetune',
    exist_ok=True,
    verbose=True,
)

stage2_best = MODEL_OUT_DIR / 'stage2_finetune' / 'weights' / 'best.pt'
print(f'\nStage 2 done. Best weights: {stage2_best}')
print(f'Stage 2 mAP50: {results_s2.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Evaluation

In [ ]:
model_eval = YOLO(str(stage2_best))
metrics = model_eval.val(data=str(YAML_PATH), imgsz=IMG_SIZE, batch=BATCH)

print('\n--- Per-class results ---')
box = metrics.box
for i, cls in enumerate(CLASSES):
    try:
        p  = box.p[i]
        r  = box.r[i]
        f1 = 2 * p * r / max(1e-8, p + r)
        ap = box.ap50[i]
        print(f'{cls:15}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={ap:.3f}')
    except (IndexError, AttributeError):
        print(f'{cls:15}  (no detections)')

print(f'\nmAP50:    {box.map50:.3f}')
print(f'mAP50-95: {box.map:.3f}')

# Save results summary
summary = {
    'model': str(stage2_best),
    'classes': CLASSES,
    'mAP50': float(box.map50),
    'mAP50_95': float(box.map),
    'per_class': {}
}
for i, cls in enumerate(CLASSES):
    try:
        p = float(box.p[i]); r = float(box.r[i])
        summary['per_class'][cls] = {
            'precision': p, 'recall': r,
            'f1': 2*p*r/max(1e-8, p+r),
            'ap50': float(box.ap50[i])
        }
    except (IndexError, AttributeError):
        summary['per_class'][cls] = None

(MODEL_OUT_DIR / 'results.json').write_text(json.dumps(summary, indent=2))
print(f'Saved to {MODEL_OUT_DIR}/results.json')


## Inference — SAHI Tiled Prediction

Uses SAHI to slice each full image into overlapping 640×640 tiles.
Detects small insects (~30px) that would be missed at full-image scale.

Install SAHI if needed: `pip install sahi`


In [ ]:
# !pip install sahi --quiet

import cv2
import csv
import matplotlib.pyplot as plt
from pathlib import Path
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


def load_sahi_model(weights_path, conf=SAHI_CONF):
    """Load trained YOLO model wrapped in SAHI for tiled inference."""
    model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=str(weights_path),
        confidence_threshold=conf,
        device='cuda:0',
    )
    print(f'SAHI model loaded: {weights_path}')
    return model


def predict_image_sahi(sahi_model, img_path, visualize=True):
    """Run SAHI tiled inference on a single full image."""
    result = get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=SAHI_SLICE_SIZE,
        slice_width=SAHI_SLICE_SIZE,
        overlap_height_ratio=SAHI_OVERLAP,
        overlap_width_ratio=SAHI_OVERLAP,
        postprocess_match_threshold=SAHI_IOU,
        verbose=0,
    )
    detections = result.object_prediction_list
    print(f'\n{Path(img_path).name}: {len(detections)} detections')
    rows = []
    for det in detections:
        cls_name = det.category.name
        conf_val = det.score.value
        bbox = det.bbox  # BoundingBox with minx, miny, maxx, maxy
        w = bbox.maxx - bbox.minx
        h = bbox.maxy - bbox.miny
        print(f'  {cls_name:15} conf={conf_val:.2f}  '
              f'size={w:.0f}x{h:.0f}px  '
              f'bbox=[{bbox.minx:.0f},{bbox.miny:.0f},{bbox.maxx:.0f},{bbox.maxy:.0f}]')
        rows.append({
            'class': cls_name, 'confidence': round(conf_val, 4),
            'x1': round(bbox.minx), 'y1': round(bbox.miny),
            'x2': round(bbox.maxx), 'y2': round(bbox.maxy),
            'w': round(w), 'h': round(h),
        })
    if visualize and detections:
        img = cv2.imread(str(img_path))
        for det in detections:
            b = det.bbox
            color = {'bumblebee': (0,165,255), 'fly': (0,255,0),
                     'butterfly': (255,0,255), 'other': (255,255,0)}.get(
                         det.category.name, (200,200,200))
            cv2.rectangle(img, (int(b.minx), int(b.miny)),
                          (int(b.maxx), int(b.maxy)), color, 2)
            cv2.putText(img, f"{det.category.name} {det.score.value:.2f}",
                        (int(b.minx), int(b.miny) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(14, 8))
        plt.imshow(img_rgb); plt.axis('off')
        plt.title(Path(img_path).name); plt.tight_layout(); plt.show()
    return rows


def predict_folder_sahi(sahi_model, img_folder, out_csv=None):
    """Run SAHI inference on all images in a folder. Saves results to CSV."""
    imgs = sorted(list(Path(img_folder).glob('*.jpg')) +
                  list(Path(img_folder).glob('*.png')))
    all_rows = []
    for img_path in imgs:
        rows = predict_image_sahi(sahi_model, img_path, visualize=False)
        for row in rows:
            row['image'] = img_path.name
        all_rows.extend(rows)
    if out_csv:
        with open(out_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=[
                'image', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2', 'w', 'h'])
            writer.writeheader(); writer.writerows(all_rows)
        print(f'\nResults saved to {out_csv}')
    print(f'\nTotal: {len(all_rows)} detections across {len(imgs)} images')
    for cls in CLASSES:
        n = sum(1 for r in all_rows if r['class'] == cls)
        print(f'  {cls:15}: {n}')
    return all_rows


# ── Example usage ────────────────────────────────────────────────────────
stage2_best = MODEL_OUT_DIR / 'stage2_finetune' / 'weights' / 'best.pt'
if stage2_best.exists():
    sahi_model = load_sahi_model(stage2_best)
    # Test on one val image
    val_imgs = list((DATASET_ROOT / 'images' / 'val').glob('*.jpg'))
    if val_imgs:
        predict_image_sahi(sahi_model, val_imgs[0])
    # Batch inference example:
    # predict_folder_sahi(sahi_model, 'path/to/folder', out_csv='detections.csv')
else:
    print(f'No checkpoint at {stage2_best} — run training first')
